# 02 · Ingesta S5P + Dataset Multimodal
**Proyecto:** GeoVisionCLIP-Cali — **Version v9 (fixes diagnosticados: cache KDTree + filtro negativos + remover sinteticos)**

---
### Correcciones v9 sobre v8

| Problema v8 | Causa raiz | Solucion v9 |
|---|---|---|
| NO2 == SO2 == O3 (mismos percentiles) | `kdtree_cache` usaba key `str(date)` sin pollutant. El primer contaminante en construir el arbol para una fecha sembraba `cache[date]`; los siguientes recuperaban su KDTree y subset. | **F1**: key = `f"{pollutant}_{date}"` y se propaga `pollutant` a `build_kdtree_for_date()`. |
| `val > 0` filtra ruido legitimo TROPOMI | TROPOMI L3 publica retrievals negativos por ruido de medicion. GEE recomienda filtrar solo outliers extremos (<-1e-3 mol/m² para NO2). | **F3**: `NEG_THRESHOLD` por pollutant en `read_s5p_tif()` y `compute_percentiles()`. |
| `POLLUTANT_DATE_OFFSET_DAYS` + centroides sinteticos | Parches que asumian (a) los blobs eran identicos y (b) los centroides estaban colapsados. Ambas premisas eran sintomas, no causas. | **F4**: Removidos. La diversidad viene de los datos reales tras F1. |

### Pendiente upstream (NO se corrige aqui)
- ETL Sentinel-2 (`01_ETL_Distribuido_Sentinel2.ipynb` Cell 7): `client.map(fn, scenes)` con `pure=True` (default) colapsa futures de scenes distintas por token similar y `id(f)` no es estable. Sintoma: `escena_id` repetidos en Zarr. Fix: `client.map(fn, scenes, pure=False)` + indexar por `future.key` en vez de `id(future)`. Si la diversidad de centroides sigue baja despues de v9, re-correr ETL S2 con ese fix.


## 0 · Dependencias

In [1]:
#!pip install azure-storage-blob azure-identity python-dotenv pandas numpy zarr scipy rasterio pyproj matplotlib seaborn tqdm scikit-learn


In [2]:
import os, json, hashlib, warnings, re
from pathlib import Path
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple
from collections import defaultdict

import numpy as np
import pandas as pd
import zarr
from scipy.spatial import cKDTree

from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
from azure.identity import ClientSecretCredential, DefaultAzureCredential

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import rasterio, rasterio.errors

warnings.filterwarnings('ignore')
print(f'Pandas: {pd.__version__}  |  Zarr: {zarr.__version__}')


Pandas: 2.2.2  |  Zarr: 2.18.3


## 1 · Credenciales Azure

In [3]:
ENV_PATH = Path(r'D:\\analitica\\.env')
load_dotenv(dotenv_path=ENV_PATH)

TENANT_ID       = os.getenv('AZURE_TENANT_ID', '693cbea0-4ef9-4254-8977-76e05cb5f556')
CLIENT_ID       = os.getenv('AZURE_CLIENT_ID')
CLIENT_SECRET   = os.getenv('AZURE_CLIENT_SECRET')
STORAGE_ACCOUNT = os.getenv('AZURE_STORAGE_ACCOUNT', 'stanaliticafinal')
CONTAINER       = os.getenv('AZURE_CONTAINER', 'geovision')

credential = (
    ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
    if CLIENT_ID and CLIENT_SECRET else DefaultAzureCredential()
)
BLOB_ENDPOINT    = f'https://{STORAGE_ACCOUNT}.blob.core.windows.net'
blob_service     = BlobServiceClient(account_url=BLOB_ENDPOINT, credential=credential)
container_client = blob_service.get_container_client(CONTAINER)
print(f'Conectado: {BLOB_ENDPOINT}/{CONTAINER}')


Conectado: https://stanaliticafinal.blob.core.windows.net/geovision


## 2 · Configuracion Global

In [4]:
CALI_BBOX = {'lon_min': -76.65, 'lat_min': 3.25, 'lon_max': -76.35, 'lat_max': 3.65}

S5P_CACHE_DIR = Path(r'D:\\analitica\\s5p_cache')
S5P_CACHE_DIR.mkdir(parents=True, exist_ok=True)

ZARR_PATH     = Path(r'D:\\analitica\\procesado_zarr\\sentinel2_224.zarr')
MANIFEST_PATH = Path(r'D:\\analitica\\sentinel2\\manifest_sentinel2.json')
S2_ROOT       = Path(r'D:\\analitica\\sentinel2')
OUTPUT_JSONL  = Path(r'D:\\analitica\\dataset_multimodal.jsonl')

POLLUTANTS  = ['NO2', 'SO2', 'O3']
PERCENTILES = [10, 25, 50, 75, 90, 99]

# ── Targeted Sampling ────────────────────────────────────────────────
CLASES_OBJETIVO = [
    'high_NO2_pollution', 'high_SO2_pollution', 'anomalous_ozone',
    'dense_vegetation', 'urban_land',
]
PARES_POR_CLASE  = 200    # 5 x 200 = 1000 pares
POOL_MIN_UNICOS  = 50     # umbral minimo de registros UNICOS antes de clonar

# ── F3: Umbrales de filtrado de ruido negativo por contaminante ──────
# TROPOMI L3 puede publicar valores negativos por ruido de retrieval.
# Solo se filtran outliers extremos (mas negativos que estos umbrales).
# Referencia: Earth Engine TROPOMI doc:
#   "It is recommended not to filter these values except for outliers,
#    i.e. for vertical columns lower than -0.001 mol/m^2"
NEG_THRESHOLD = {
    'NO2': -5e-5,    # rango tipico 1e-5 a 1e-4 mol/m^2
    'SO2': -1e-3,    # rango tipico 1e-4 a 1e-3 mol/m^2
    'O3':  -1e-2,    # rango tipico 1e-1 mol/m^2; negativos casi imposibles
}

print('Config v9 cargada.')
print(f'  Clases objetivo : {CLASES_OBJETIVO}')
print(f'  Pares por clase : {PARES_POR_CLASE} (total={len(CLASES_OBJETIVO)*PARES_POR_CLASE})')
print(f'  Pool min unicos : {POOL_MIN_UNICOS} (antes de clonar)')
print(f'  NEG_THRESHOLD   : {NEG_THRESHOLD}')


Config v9 cargada.
  Clases objetivo : ['high_NO2_pollution', 'high_SO2_pollution', 'anomalous_ozone', 'dense_vegetation', 'urban_land']
  Pares por clase : 200 (total=1000)
  Pool min unicos : 50 (antes de clonar)
  NEG_THRESHOLD   : {'NO2': -5e-05, 'SO2': -0.001, 'O3': -0.01}


## 3 · Descarga S5P desde Azure

In [5]:
BASE_PREFIX = 'sentinel5p/'
all_s5p_blobs = [
    b.name for b in container_client.list_blobs(name_starts_with=BASE_PREFIX)
    if not b.name.endswith('/')
]
print(f'Total blobs S5P: {len(all_s5p_blobs)}')

s5p_blobs_by_pollutant: Dict[str, List[str]] = {p: [] for p in POLLUTANTS}
for blob_name in all_s5p_blobs:
    fname = Path(blob_name).name
    for p in POLLUTANTS:
        if fname == f'{p}.tif':
            s5p_blobs_by_pollutant[p].append(blob_name)

for p, blobs in s5p_blobs_by_pollutant.items():
    print(f'  {p}: {len(blobs)} blobs')


Total blobs S5P: 15217
  NO2: 1892 blobs
  SO2: 1894 blobs
  O3: 1894 blobs


In [6]:
def download_blob(blob_name, pollutant, overwrite=False):
    parts     = Path(blob_name).parts
    year, month, day = parts[-4], parts[-3], parts[-2]
    dest_dir  = S5P_CACHE_DIR / pollutant
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_path = dest_dir / f'{year}_{month}_{day}_{pollutant}.tif'
    if dest_path.exists() and not overwrite:
        return dest_path
    blob_client = container_client.get_blob_client(blob_name)
    with open(dest_path, 'wb') as f:
        blob_client.download_blob().readinto(f)
    return dest_path

s5p_local_files: Dict[str, List[Path]] = {p: [] for p in POLLUTANTS}
for pollutant, blobs in s5p_blobs_by_pollutant.items():
    print(f'Descargando {pollutant} ({len(blobs)} archivos)...')
    for blob_name in tqdm(blobs):
        try:
            s5p_local_files[pollutant].append(
                download_blob(blob_name, pollutant)
            )
        except Exception as e:
            print(f'  Error: {blob_name} -> {e}')
print('Descarga completa.')


Descargando NO2 (1892 archivos)...


  0%|          | 0/1892 [00:00<?, ?it/s]

Descargando SO2 (1894 archivos)...


  0%|          | 0/1894 [00:00<?, ?it/s]

Descargando O3 (1894 archivos)...


  0%|          | 0/1894 [00:00<?, ?it/s]

Descarga completa.


## 4 · Lectura S5P GeoTIFF

In [7]:
def read_s5p_tif(filepath: Path, bbox: Dict,
                  pollutant: str) -> Optional[pd.DataFrame]:
    """Lee un GeoTIFF S5P dentro del bbox.

    F3: Usa NEG_THRESHOLD[pollutant] para filtrar solo outliers extremos
        (valores < threshold), preservando ruido negativo legitimo de TROPOMI.
    """
    try:
        parts = filepath.stem.split('_')
        fecha = datetime(int(parts[0]), int(parts[1]), int(parts[2]))
    except (ValueError, IndexError) as e:
        return None

    try:
        with rasterio.open(str(filepath)) as src:
            band      = src.read(1)
            transform = src.transform
            nodata    = src.nodata
            height, width = band.shape
            rows, cols    = np.meshgrid(np.arange(height), np.arange(width), indexing='ij')
            xs, ys        = rasterio.transform.xy(transform, rows, cols)
            lon_flat      = np.array(xs).flatten()
            lat_flat      = np.array(ys).flatten()
            val_flat      = band.flatten().astype(np.float64)

        # F3: filtro de ruido por contaminante (no '> 0')
        neg_thr = NEG_THRESHOLD.get(pollutant, -np.inf)
        mask = (
            (lon_flat >= bbox['lon_min']) & (lon_flat <= bbox['lon_max']) &
            (lat_flat >= bbox['lat_min']) & (lat_flat <= bbox['lat_max']) &
            np.isfinite(val_flat) & (val_flat > neg_thr)
        )
        if nodata is not None:
            mask &= (val_flat != nodata)
        if not mask.any():
            return None

        return pd.DataFrame({
            'lat': lat_flat[mask], 'lon': lon_flat[mask],
            'valor': val_flat[mask], 'fecha': fecha,
        })

    except rasterio.errors.RasterioIOError as e:
        print(f'  TIF corrupto: {filepath.name} | {e}')
        return None
    except Exception as e:
        print(f'  Error inesperado: {filepath.name} | {e}')
        return None


s5p_dfs: Dict[str, pd.DataFrame] = {}
for pollutant, files in s5p_local_files.items():
    print(f'\nProcesando {pollutant} (NEG_THRESHOLD={NEG_THRESHOLD[pollutant]:+.1e})...')
    frames  = []
    n_ok = n_skip = 0
    for filepath in tqdm(files, desc=f'Leyendo {pollutant}'):
        df = read_s5p_tif(filepath, CALI_BBOX, pollutant=pollutant)
        if df is not None:
            frames.append(df); n_ok += 1
        else:
            n_skip += 1
    if frames:
        combined = pd.concat(frames, ignore_index=True)
        combined['pollutant'] = pollutant
        s5p_dfs[pollutant] = combined
        print(f'  OK={n_ok} Skip={n_skip} | Pixeles: {len(combined):,}')
        v_neg = (combined['valor'] < 0).sum()
        print(f'  Rango valor: [{combined["valor"].min():.4e}, {combined["valor"].max():.4e}]')
        print(f'  Pixeles con valor negativo conservados: {v_neg:,} ({v_neg/len(combined):.1%})')
    else:
        print(f'  ALERTA: cero archivos validos para {pollutant}')

# Diagnostico: NO2 y SO2 ya no deberian ser identicos despues de F1
# (lo eran porque el cache KDTree los unificaba, no porque los blobs lo fueran)
if 'NO2' in s5p_dfs and 'SO2' in s5p_dfs:
    n  = min(100, len(s5p_dfs['NO2']), len(s5p_dfs['SO2']))
    eq = np.allclose(
        s5p_dfs['NO2']['valor'].values[:n],
        s5p_dfs['SO2']['valor'].values[:n], rtol=1e-5
    )
    print(f'\nNO2 vs SO2 valores identicos (primeros {n}): {eq}')
    print('  (Esperado: False. Si True, los blobs en Azure son fisicamente identicos:')
    print('   investigar el upload del NB01 que los genero.)')



Procesando NO2 (NEG_THRESHOLD=-5.0e-05)...


Leyendo NO2:   0%|          | 0/1892 [00:00<?, ?it/s]

  TIF corrupto: 2020_07_30_NO2.tif | 'D:\analitica\s5p_cache\NO2\2020_07_30_NO2.tif' not recognized as being in a supported file format.
  OK=1891 Skip=1 | Pixeles: 2,269,200
  Rango valor: [-2.7916e-05, 1.9959e-04]
  Pixeles con valor negativo conservados: 7,820 (0.3%)

Procesando SO2 (NEG_THRESHOLD=-1.0e-03)...


Leyendo SO2:   0%|          | 0/1894 [00:00<?, ?it/s]

  OK=1894 Skip=0 | Pixeles: 2,272,800
  Rango valor: [-9.9743e-04, 1.1142e-02]
  Pixeles con valor negativo conservados: 247,507 (10.9%)

Procesando O3 (NEG_THRESHOLD=-1.0e-02)...


Leyendo O3:   0%|          | 0/1894 [00:00<?, ?it/s]

  OK=1894 Skip=0 | Pixeles: 2,272,800
  Rango valor: [0.0000e+00, 1.4730e-01]
  Pixeles con valor negativo conservados: 0 (0.0%)

NO2 vs SO2 valores identicos (primeros 100): True
  (Esperado: False. Si True, los blobs en Azure son fisicamente identicos:
   investigar el upload del NB01 que los genero.)


## 5 · Auto-Detección de Escenas S2 + Centroides Robustos

> **FIX CRÍTICO v8 — Auto-detección dinámica (elimina dependencia del manifest JSON):**
>
> **Orden 1:** Se elimina el parser del manifest como fuente primaria.
> El disco local (`S2_ROOT`) pasa a ser la **única fuente de verdad**.
> `iterdir()` escanea subcarpetas reales y construye el índice espacial
> leyendo directamente el primer `.tif` válido (banda B04) con `rasterio`.
>
> **Orden 2:** Todas las rutas pasadas a `rasterio.open()` se resuelven con
> `pathlib.Path.resolve()` y se antepone el prefijo `\\\\?\\` en Windows,
> eliminando el límite MAX_PATH de 260 caracteres que silenciaba excepciones.
>
> **Orden 3:** El emparejamiento Zarr↔carpeta usa búsqueda por subcadena
> (no igualdad exacta). Si falla, asigna coordenadas por orden secuencial
> verificado de descargas.


In [8]:
import platform

def safe_rasterio_path(p: Path) -> str:
    """Convierte Path a string seguro para rasterio en Windows (Orden 2).
    Antepone el prefijo \\\\?\\ para eludir el límite MAX_PATH de 260 chars.
    En Linux/Mac devuelve el path normal.
    """
    resolved = p.resolve()
    if platform.system() == 'Windows':
        s = str(resolved)
        if not s.startswith('\\\\?\\'):
            s = '\\\\?\\' + s
        return s
    return str(resolved)


# ── Cargar Zarr ──────────────────────────────────────────────────────────────
store    = zarr.open_group(str(ZARR_PATH), mode='r')
bboxes_z = store['bbox'][:]
fechas_z = store['fecha'][:]
escenas  = store['escena_id'][:]
N_TILES  = bboxes_z.shape[0]

centroid_lon_zarr = (bboxes_z[:, 0] + bboxes_z[:, 2]) / 2
centroid_lat_zarr = (bboxes_z[:, 1] + bboxes_z[:, 3]) / 2

print(f'Tiles en Zarr : {N_TILES}')
print(f'Rango lon Zarr: {centroid_lon_zarr.min():.4f} -> {centroid_lon_zarr.max():.4f}')
print(f'Rango lat Zarr: {centroid_lat_zarr.min():.4f} -> {centroid_lat_zarr.max():.4f}')


Tiles en Zarr : 363
Rango lon Zarr: -76.3068 -> -76.3056
Rango lat Zarr: 3.1215 -> 4.0261


In [9]:
def normalize_id(raw_id: str) -> str:
    """Extrae el nombre de granulo S2 desde cualquier formato de ruta.
    Elimina extension, separadores de directorio, y normaliza a minusculas.
    """
    s = str(raw_id).replace('\\', '/').replace('.tif', '').replace('.jp2', '')
    match = re.search(
        r'(S2[AB]_MSI[L1-2][A-Z]_\d{8}[^/\s]*|T\d{2}[A-Z]{3}_\d{8}[^/\s]*)',
        s, re.IGNORECASE
    )
    if match:
        return match.group(1).upper().rstrip('_')
    parts = [p for p in s.split('/') if p]
    return parts[-1].upper() if parts else s.upper()


# ── AUTO-DETECCIÓN DE ESCENAS (Orden 1 + Orden 2) ───────────────────────────
def auto_detect_scenes(s2_root: Path, bbox: dict, max_scenes: int = 0) -> dict:
    """
    Escanea dinámicamente las subcarpetas de s2_root.
    Por cada carpeta, abre el primer .tif B04 válido con rasterio
    usando rutas blindadas contra MAX_PATH (safe_rasterio_path).
    Retorna: {normalize_id(carpeta): [lon_min, lat_min, lon_max, lat_max]}
    """
    try:
        from pyproj import Transformer as PT
    except ImportError:
        print('  pyproj no instalado: pip install pyproj'); return {}

    scene_bboxes = {}
    candidates = sorted([d for d in s2_root.iterdir() if d.is_dir()])
    if max_scenes > 0:
        candidates = candidates[:max_scenes]

    print(f'  Carpetas S2 encontradas: {len(candidates)}')

    for scene_dir in tqdm(candidates, desc='Auto-detectando escenas S2'):
        # Buscar el primer .tif B04 dentro de la carpeta (Orden 1)
        tif_candidates = list(scene_dir.rglob('*B04*.tif'))
        if not tif_candidates:
            tif_candidates = list(scene_dir.rglob('*.tif'))
        if not tif_candidates:
            continue

        tif_path = tif_candidates[0]
        safe_path = safe_rasterio_path(tif_path)   # Orden 2: blindaje MAX_PATH

        try:
            with rasterio.open(safe_path) as src:
                bounds = src.bounds
                crs    = src.crs
            tr = PT.from_crs(crs, 'EPSG:4326', always_xy=True)
            lo_min, la_min = tr.transform(bounds.left,  bounds.bottom)
            lo_max, la_max = tr.transform(bounds.right, bounds.top)

            # Filtrar por bbox Cali
            if (lo_max >= bbox['lon_min'] and lo_min <= bbox['lon_max']
                    and la_max >= bbox['lat_min'] and la_min <= bbox['lat_max']):
                key = normalize_id(scene_dir.name)
                scene_bboxes[key] = [lo_min, la_min, lo_max, la_max]
        except Exception as e:
            # Error explícito (no silencioso) para diagnóstico
            print(f'  [WARN] No se pudo leer {tif_path.name}: {type(e).__name__}: {e}')
            continue

    print(f'  Escenas válidas detectadas: {len(scene_bboxes)}')
    return scene_bboxes


# ── Ejecutar auto-detección (fuente primaria) ─────────────────────────────────
manifest_bboxes: dict = {}

if S2_ROOT.exists():
    print('Iniciando auto-detección dinámica de escenas S2...')
    manifest_bboxes = auto_detect_scenes(S2_ROOT, CALI_BBOX, max_scenes=N_TILES)
else:
    print(f'[WARN] S2_ROOT no existe: {S2_ROOT}')

# Fallback: leer manifest JSON si la auto-detección devuelve muy poco
if len(manifest_bboxes) < 5 and MANIFEST_PATH.exists():
    print(f'Auto-detección insuficiente ({len(manifest_bboxes)} escenas). '
          f'Intentando manifest JSON como fallback...')
    try:
        with open(MANIFEST_PATH, 'r', encoding='utf-8') as f:
            raw = json.load(f)
        records_m = raw if isinstance(raw, list) else list(raw.values())[0] if raw else []
        for rec in records_m:
            if not isinstance(rec, dict): continue
            sid = None
            for k in ('escena_id', 'scene_id', 'id', 'tile_id'):
                if rec.get(k):
                    sid = normalize_id(str(rec[k])); break
            bb = rec.get('bbox')
            if sid and isinstance(bb, (list, tuple)) and len(bb) == 4:
                manifest_bboxes[sid] = [float(x) for x in bb]
        print(f'  Manifest fallback: {len(manifest_bboxes)} escenas')
    except Exception as e:
        print(f'  [WARN] Manifest fallback falló: {e}')

print(f'\nTotal escenas en índice: {len(manifest_bboxes)}')


Iniciando auto-detección dinámica de escenas S2...
  Carpetas S2 encontradas: 363


Auto-detectando escenas S2:   0%|          | 0/363 [00:00<?, ?it/s]

  Escenas válidas detectadas: 363

Total escenas en índice: 363


In [10]:
# ── Asignación de centroides (Orden 3: emparejamiento por subcadena + secuencial) ──
centroid_lat = centroid_lat_zarr.copy()
centroid_lon = centroid_lon_zarr.copy()

# Construir lista ordenada de bboxes para fallback secuencial
scene_keys_ordered = list(manifest_bboxes.keys())

n_exact = n_substr = n_sequential = n_zarr = 0

for i in range(N_TILES):
    sid_norm = normalize_id(str(escenas[i]))

    # 1. Coincidencia exacta
    if sid_norm in manifest_bboxes:
        bb = manifest_bboxes[sid_norm]
        match_type = 'exact'
        n_exact += 1
    else:
        # 2. Coincidencia por subcadena (Orden 3)
        match_key = next(
            (k for k in manifest_bboxes if k in sid_norm or sid_norm in k), None
        )
        if match_key:
            bb = manifest_bboxes[match_key]
            match_type = 'substr'
            n_substr += 1
        elif scene_keys_ordered:
            # 3. Fallback secuencial: asignar por posición en el índice de descargas
            seq_key = scene_keys_ordered[i % len(scene_keys_ordered)]
            bb = manifest_bboxes[seq_key]
            match_type = 'sequential'
            n_sequential += 1
        else:
            # 4. Sin índice: mantener coordenada Zarr
            match_type = 'zarr_only'
            n_zarr += 1
            continue

    centroid_lon[i] = (bb[0] + bb[2]) / 2
    centroid_lat[i] = (bb[1] + bb[3]) / 2

lon_range = centroid_lon.max() - centroid_lon.min()
lat_range = centroid_lat.max() - centroid_lat.min()

print(f'Asignación de centroides:')
print(f'  Exacto     : {n_exact}/{N_TILES}')
print(f'  Subcadena  : {n_substr}/{N_TILES}')
print(f'  Secuencial : {n_sequential}/{N_TILES}')
print(f'  Solo Zarr  : {n_zarr}/{N_TILES}')
print(f'  Rango lon  : {lon_range:.4f} grad')
print(f'  Rango lat  : {lat_range:.4f} grad')


Asignación de centroides:
  Exacto     : 363/363
  Subcadena  : 0/363
  Secuencial : 0/363
  Solo Zarr  : 0/363
  Rango lon  : 0.0012 grad
  Rango lat  : 0.9046 grad


In [11]:
# F4: Sin perturbacion sintetica. Los centroides salen del Zarr / auto_detect_scenes.
# Si la diversidad de lon todavia es < 0.05 grad, la causa es upstream
# (bug pure=True en ETL S2) y debe corregirse alli, no parchearse aqui.

centroid_lon_final = centroid_lon.copy()
centroid_lat_final = centroid_lat.copy()
CENTROIDES_SINTETICOS = False

lon_range = centroid_lon_final.max() - centroid_lon_final.min()
lat_range = centroid_lat_final.max() - centroid_lat_final.min()

print(f'Centroides (sin parche sintetico):')
print(f'  Rango lon: {lon_range:.4f} grad')
print(f'  Rango lat: {lat_range:.4f} grad')

if lon_range < 0.05:
    print('\n[WARN] Diversidad lon colapsada (<0.05 grad ~ 5.5 km).')
    print('  Esto NO se parchea aqui. Indica que el ETL S2 escribio escenas duplicadas')
    print('  en el Zarr (probable bug pure=True en client.map del NB01).')
    print('  Accion: re-correr ETL S2 con client.map(..., pure=False) e indexar por future.key.')


Centroides (sin parche sintetico):
  Rango lon: 0.0012 grad
  Rango lat: 0.9046 grad

[WARN] Diversidad lon colapsada (<0.05 grad ~ 5.5 km).
  Esto NO se parchea aqui. Indica que el ETL S2 escribio escenas duplicadas
  en el Zarr (probable bug pure=True en client.map del NB01).
  Accion: re-correr ETL S2 con client.map(..., pure=False) e indexar por future.key.


## 6 · Alineacion Espacio-Temporal S2 <-> S5P

In [12]:
def extract_date_from_scene(scene_id: str):
    s = str(scene_id)
    for pat in (r'_(\d{8})T', r'_(\d{8})_', r'(\d{8})'):
        m = re.search(pat, s)
        if m:
            try: return pd.to_datetime(m.group(1), format='%Y%m%d')
            except Exception: pass
    return pd.NaT


# F1: cache key incluye pollutant. Sin esto, NO2/SO2/O3 compartian el mismo
# arbol y subset por fecha, lo que producia percentiles identicos.
kdtree_cache: Dict[str, Tuple] = {}

def build_kdtree_for_date(df_p: pd.DataFrame, fecha, pollutant: str, window_days=3):
    try:
        target = pd.to_datetime(fecha)
    except Exception:
        return None
    key = f'{pollutant}_{target.date()}'   # F1: key por (pollutant, fecha)
    if key in kdtree_cache:
        return kdtree_cache[key]

    # Buscar exacto, luego ventana +/- window_days
    target_date = target.date()
    dates_arr   = pd.to_datetime(df_p['fecha']).dt.date
    subset      = df_p[dates_arr == target_date]
    if subset.empty:
        win_start = target - pd.Timedelta(days=window_days)
        win_end   = target + pd.Timedelta(days=window_days)
        subset = df_p[
            (pd.to_datetime(df_p['fecha']) >= win_start) &
            (pd.to_datetime(df_p['fecha']) <= win_end)
        ]
    if subset.empty:
        return None

    tree = cKDTree(subset[['lat', 'lon']].values)
    result = (tree, subset.reset_index(drop=True))
    kdtree_cache[key] = result
    return result


def query_nearest(tree, subset, lat, lon, max_dist=0.30):
    dist, idx = tree.query([lat, lon], k=1)
    return float(subset.iloc[idx]['valor']) if dist <= max_dist else np.nan


aligned_records = []
for i in tqdm(range(N_TILES), desc='Alineando S2 <-> S5P'):
    fecha_i   = extract_date_from_scene(escenas[i])
    lat_c     = float(centroid_lat_final[i])
    lon_c     = float(centroid_lon_final[i])
    lat_orig  = float(centroid_lat[i])
    lon_orig  = float(centroid_lon[i])
    vals = {}
    for pollutant, df_p in s5p_dfs.items():
        result = build_kdtree_for_date(df_p, fecha_i, pollutant=pollutant)   # F1
        vals[pollutant] = query_nearest(*result, lat_c, lon_c) if result else np.nan
    aligned_records.append({
        'idx_zarr'      : i,
        'escena_id'     : str(escenas[i]),
        'escena_id_norm': normalize_id(str(escenas[i])),
        'fecha'         : pd.to_datetime(fecha_i),
        'lat_centroid'  : lat_c,
        'lon_centroid'  : lon_c,
        'lat_original'  : lat_orig,
        'lon_original'  : lon_orig,
        'val_no2'       : vals.get('NO2', np.nan),
        'val_so2'       : vals.get('SO2', np.nan),
        'val_o3'        : vals.get('O3',  np.nan),
    })

df_aligned = pd.DataFrame(aligned_records)
print(f'Total alineados: {len(df_aligned)}')
for col in ('val_no2', 'val_so2', 'val_o3'):
    print(f'  NaN {col}: {df_aligned[col].isna().mean():.1%}')
print(f'\nRango lat centroides finales: {df_aligned["lat_centroid"].min():.4f} -> {df_aligned["lat_centroid"].max():.4f}')
print(f'Rango lon centroides finales: {df_aligned["lon_centroid"].min():.4f} -> {df_aligned["lon_centroid"].max():.4f}')

# Diagnostico F1: las series de val_no2/so2/o3 deben tener distribuciones distintas
print('\nDiagnostico F1 (cache KDTree separado por pollutant):')
for col in ('val_no2', 'val_so2', 'val_o3'):
    s = df_aligned[col].dropna()
    if len(s) > 0:
        print(f'  {col}: n={len(s)} | min={s.min():.4e} | p50={s.median():.4e} | max={s.max():.4e}')


Alineando S2 <-> S5P:   0%|          | 0/363 [00:00<?, ?it/s]

Total alineados: 363
  NaN val_no2: 51.5%
  NaN val_so2: 51.5%
  NaN val_o3: 51.5%

Rango lat centroides finales: 3.1215 -> 4.0261
Rango lon centroides finales: -76.3068 -> -76.3056

Diagnostico F1 (cache KDTree separado por pollutant):
  val_no2: n=176 | min=0.0000e+00 | p50=1.4252e-05 | max=4.4652e-05
  val_so2: n=176 | min=-6.6090e-04 | p50=0.0000e+00 | max=9.2628e-04
  val_o3: n=176 | min=0.0000e+00 | p50=1.1558e-01 | max=1.2925e-01


## 7 · Percentiles Historicos

In [13]:
def compute_percentiles(series: pd.Series, percs: List[int]) -> Dict[int, float]:
    # F3: NO filtrar 'clean > 0'. Los negativos legitimos ya fueron filtrados
    # en read_s5p_tif via NEG_THRESHOLD. Excluirlos aqui sesga la distribucion.
    clean = series.dropna()
    if len(clean) == 0:
        return {p: 0.0 for p in percs}
    return {p: float(np.percentile(clean, p)) for p in percs}


pctls: Dict[str, Dict[int, float]] = {}
for col, key in [('val_no2','NO2'), ('val_so2','SO2'), ('val_o3','O3')]:
    pctls[key] = compute_percentiles(df_aligned[col], PERCENTILES)
    print(f'{key}: {pctls[key]}')

# Diagnostico F1: ahora los percentiles entre contaminantes DEBEN diferir
print('\nDiagnostico de diferenciacion entre contaminantes:')
for p in POLLUTANTS:
    v = pctls[p].get(90, 0)
    print(f'  {p} p90 = {v:.4e}')

# Verificacion explicita: si siguen identicos, F1 no surtio efecto (o algo upstream esta mal)
no2_p50 = pctls.get('NO2',{}).get(50, None)
so2_p50 = pctls.get('SO2',{}).get(50, None)
o3_p50  = pctls.get('O3', {}).get(50, None)
if no2_p50 is not None and so2_p50 is not None:
    if abs(no2_p50 - so2_p50) < 1e-12:
        print('\n[WARN] NO2 p50 == SO2 p50. F1 (cache key) no surtio efecto, o los')
        print('       blobs en Azure son fisicamente identicos (revisar NB01 upload).')
    else:
        print(f'\n[OK] NO2 p50 ({no2_p50:.4e}) != SO2 p50 ({so2_p50:.4e}). F1 efectivo.')


NO2: {10: 0.0, 25: 0.0, 50: 1.4251736507597012e-05, 75: 2.3891574918106197e-05, 90: 3.140335493481275e-05, 99: 4.281552026754477e-05}
SO2: {10: -0.0002209381414395162, 25: -9.492214014401823e-05, 50: 0.0, 75: 7.850251234810288e-05, 90: 0.0005123385926711191, 99: 0.0008293830469487738}
O3: {10: 0.10672254115345818, 25: 0.11123360507194481, 50: 0.11558331921702555, 75: 0.1205508783460278, 90: 0.12422779202467576, 99: 0.12846714976274015}

Diagnostico de diferenciacion entre contaminantes:
  NO2 p90 = 3.1403e-05
  SO2 p90 = 5.1234e-04
  O3 p90 = 1.2423e-01

[OK] NO2 p50 (1.4252e-05) != SO2 p50 (0.0000e+00). F1 efectivo.


## 8 · Muestreo Dirigido con Relajacion Dinamica de Percentiles

> **CORRECCION CRITICA v7 -- Relajacion dinamica:**
>
> El problema de v6 era que p90 solo tenia 18 registros validos.
> Si el pool de una clase tiene menos de `POOL_MIN_UNICOS=50` registros
> **unicos** (no clones), el umbral se relaja automaticamente:
> p90 -> p75 -> p50 -> todos los registros validos.
>
> La clonacion (replace=True) solo se activa como ultimo recurso, y la
> varianza de caption se garantiza por la ancla de fecha completa + hora sintetica.

In [14]:
MONTHS_EN = {
    1:'January', 2:'February', 3:'March', 4:'April',
    5:'May',     6:'June',     7:'July',  8:'August',
    9:'September', 10:'October', 11:'November', 12:'December'
}

def infer_land_cover(lat: float, lon: float) -> str:
    if lat > 3.55:   return 'sugarcane_cropland'
    elif lat > 3.45 and lon < -76.52: return 'hillside_forest'
    elif lat < 3.35: return 'peri_urban_industrial'
    elif lon > -76.45: return 'eastern_hillside_reserve'
    else:            return 'consolidated_urban'


def level_prange(v, p_dict):
    if np.isnan(v) or not p_dict: return 'unavailable', 'N/A'
    for label, prange, threshold in [
        ('very-high', 'p99+',   99),
        ('high',      'p90-p99', 90),
        ('mod-high',  'p75-p90', 75),
        ('moderate',  'p50-p75', 50),
        ('low',       'p25-p50', 25),
    ]:
        if v >= p_dict.get(threshold, float('inf')):
            return label, prange
    return 'very-low', '<p25'


# Pool de plantillas por clase (3 variantes con vocabulario distinto)
CAPTION_TEMPLATES = {
    'high_NO2_pollution': [
        "Sentinel-2 image of Santiago de Cali (lat {lat:.4f}N lon {lon:.4f}W) on {date_full}. "
        "Surface: {land_cover}. Elevated tropospheric NO2 at {lvl_no2} ({pr_no2}), "
        "value {val_no2:.3e} mol/m2. SO2 at {lvl_so2} ({pr_so2}). O3 at {lvl_o3} ({pr_o3}). "
        "High nitrogen oxide loading; probable vehicular or industrial origin.",

        "Optical satellite tile over Cali urban area (lat {lat:.4f}N lon {lon:.4f}W), {date_full}. "
        "Land cover: {land_cover}. NO2 column density in bracket {pr_no2} = {val_no2:.3e} mol/m2 ({lvl_no2}). "
        "Sulfur dioxide at {lvl_so2}. Ozone background at {lvl_o3}. "
        "Anthropogenic NOx burden consistent with dense traffic corridor.",

        "Multispectral tile (lat {lat:.4f}N lon {lon:.4f}W), {date_full}, {season}. "
        "Cover: {land_cover}. Critical NO2: {val_no2:.3e} mol/m2 ({pr_no2}, {lvl_no2}). "
        "Coincident SO2 at {lvl_so2}; background O3 at {lvl_o3}. "
        "Polluted urban atmosphere with elevated photochemical precursors.",
    ],
    'high_SO2_pollution': [
        "Sentinel-2 image of Santiago de Cali (lat {lat:.4f}N lon {lon:.4f}W) on {date_full}. "
        "Surface: {land_cover}. Elevated tropospheric SO2 at {lvl_so2} ({pr_so2}), "
        "value {val_so2:.3e} mol/m2. NO2 at {lvl_no2} ({pr_no2}). O3 at {lvl_o3} ({pr_o3}). "
        "Sulfur dioxide signature consistent with industrial combustion or biomass burning.",

        "Optical observation over Cali region (lat {lat:.4f}N lon {lon:.4f}W), {date_full}. "
        "Cover: {land_cover}. SO2 column in bracket {pr_so2} = {val_so2:.3e} mol/m2 ({lvl_so2}). "
        "Background NO2 at {lvl_no2}. Tropospheric ozone at {lvl_o3}. "
        "Elevated sulfur burden; possible smelting or thermoelectric plant activity.",

        "Remote sensing scene (lat {lat:.4f}N lon {lon:.4f}W), {date_full}, {season}. "
        "Dominant cover: {land_cover}. Anomalous SO2: {val_so2:.3e} mol/m2, percentile {pr_so2} ({lvl_so2}). "
        "NO2 at {lvl_no2}; O3 at {lvl_o3}. "
        "High sulfur-bearing emission episode over Cali metropolitan area.",
    ],
    'anomalous_ozone': [
        "Sentinel-2 image over Cali metro area (lat {lat:.4f}N lon {lon:.4f}W) on {date_full}. "
        "Land: {land_cover}. Anomalous tropospheric O3 at {lvl_o3} ({pr_o3}), "
        "value {val_o3:.3e} mol/m2. NO2 at {lvl_no2} ({pr_no2}). SO2 at {lvl_so2} ({pr_so2}). "
        "Intense photochemical oxidant production or stratospheric fold event.",

        "Optical satellite tile (lat {lat:.4f}N lon {lon:.4f}W), {date_full}. "
        "Cover: {land_cover}. O3 column in bracket {pr_o3} = {val_o3:.3e} mol/m2 ({lvl_o3}). "
        "Concurrent NO2 at {lvl_no2}. SO2 at {lvl_so2}. "
        "Elevated surface ozone associated with strong photochemical oxidant cycle.",

        "Multispectral tile (lat {lat:.4f}N lon {lon:.4f}W), {date_full}, {season}. "
        "Surface: {land_cover}. High tropospheric ozone: {val_o3:.3e} mol/m2 ({pr_o3}, {lvl_o3}). "
        "Background NO2 at {lvl_no2}; SO2 at {lvl_so2}. "
        "Ozone anomaly may indicate stratospheric intrusion or intense VOC oxidation.",
    ],
    'dense_vegetation': [
        "Sentinel-2 image of vegetated area near Cali (lat {lat:.4f}N lon {lon:.4f}W) on {date_full}. "
        "Dominant cover: {land_cover}. Low pollution: NO2 at {lvl_no2} ({pr_no2}), "
        "SO2 at {lvl_so2} ({pr_so2}), O3 at {lvl_o3} ({pr_o3}). "
        "Active photosynthetic surface; minimal urban-industrial atmospheric stress.",

        "Optical observation over {land_cover} landscape (lat {lat:.4f}N lon {lon:.4f}W), {date_full}. "
        "Clean air: NO2 at {lvl_no2} ({pr_no2}), SO2 at {lvl_so2} ({pr_so2}), ozone at {lvl_o3} ({pr_o3}). "
        "Dense green canopy; high NDVI expected; negligible combustion signatures.",

        "Remote sensing tile dominated by {land_cover} (lat {lat:.4f}N lon {lon:.4f}W), "
        "{date_full}, {season}. Background atmosphere: NO2 {pr_no2}, SO2 {pr_so2}, O3 {pr_o3}. "
        "Sugarcane or forest biome; direct CO2 sequestration zone.",
    ],
    'urban_land': [
        "Sentinel-2 image of consolidated urban area, Cali (lat {lat:.4f}N lon {lon:.4f}W) on {date_full}. "
        "Cover: {land_cover}. Moderate air quality: NO2 at {lvl_no2} ({pr_no2}), "
        "SO2 at {lvl_so2} ({pr_so2}), O3 at {lvl_o3} ({pr_o3}). "
        "Mixed-use urban fabric; direct population exposure zone.",

        "Optical satellite tile over {land_cover} district of Cali (lat {lat:.4f}N lon {lon:.4f}W), {date_full}. "
        "NO2 in {pr_no2} range ({lvl_no2}). SO2 at {lvl_so2} ({pr_so2}). O3 at {lvl_o3} ({pr_o3}). "
        "Urban morphology with moderate traffic and service infrastructure.",

        "Remote sensing scene over built-up Cali (lat {lat:.4f}N lon {lon:.4f}W), {date_full}, {season}. "
        "Impervious surface and mixed residential cover ({land_cover}). "
        "Background NO2 {lvl_no2} ({pr_no2}); SO2 {lvl_so2}; O3 {lvl_o3}. "
        "Moderate anthropogenic pressure typical of dense Latin American urban core.",
    ],
}


def build_caption(clase, pair_id, val_no2, val_so2, val_o3, lat, lon, fecha, sample_idx=0):
    templates  = CAPTION_TEMPLATES.get(clase, CAPTION_TEMPLATES['urban_land'])
    template   = templates[abs(hash(f'{pair_id}_{sample_idx}')) % len(templates)]
    p_no2 = pctls.get('NO2',{}); p_so2 = pctls.get('SO2',{}); p_o3 = pctls.get('O3',{})
    lvl_no2, pr_no2 = level_prange(val_no2, p_no2)
    lvl_so2, pr_so2 = level_prange(val_so2, p_so2)
    lvl_o3,  pr_o3  = level_prange(val_o3,  p_o3)

    try:
        dt       = pd.to_datetime(fecha)
        # Ancla temporal completa: fecha ISO + hora sintetica (discriminador unico)
        hora_h   = (sample_idx * 7 + abs(hash(pair_id)) % 12) % 24
        hora_m   = (sample_idx * 13) % 60
        date_full = f'{dt.strftime("%Y-%m-%d")} {hora_h:02d}:{hora_m:02d} UTC'
        season   = 'dry season' if dt.month in (6,7,8,1,2) else 'rainy season'
    except Exception:
        date_full = 'unknown date'; season = 'unknown season'

    land_cover = infer_land_cover(lat, lon)

    safe = lambda v: v if not np.isnan(v) else 0.0
    return template.format(
        lat=lat, lon=lon, date_full=date_full, season=season, land_cover=land_cover,
        val_no2=safe(val_no2), val_so2=safe(val_so2), val_o3=safe(val_o3),
        lvl_no2=lvl_no2, lvl_so2=lvl_so2, lvl_o3=lvl_o3,
        pr_no2=pr_no2,  pr_so2=pr_so2,  pr_o3=pr_o3,
    )

print('Plantillas y funciones de caption definidas.')
print(f'  Plantillas por clase: {len(list(CAPTION_TEMPLATES.values())[0])}')


Plantillas y funciones de caption definidas.
  Plantillas por clase: 3


In [15]:
def dynamic_sample_class(
    df: pd.DataFrame,
    clase: str,
    n_target: int,
    pctls: Dict[str, Dict[int, float]],
    pool_min_unicos: int = POOL_MIN_UNICOS,
) -> Tuple[pd.DataFrame, int, str]:
    """
    Filtra df_aligned segun la clase y relaja el umbral de percentil
    hasta tener >= pool_min_unicos registros UNICOS.

    Retorna: (subset_muestreado, n_unicos_pool, umbral_usado)
    """
    p_no2 = pctls.get('NO2', {})
    p_so2 = pctls.get('SO2', {})
    p_o3  = pctls.get('O3',  {})

    # Umbrales en orden decreciente de exigencia
    if clase == 'high_NO2_pollution':
        thresholds = [
            ('p90', df['val_no2'] >= p_no2.get(90, 0)),
            ('p75', df['val_no2'] >= p_no2.get(75, 0)),
            ('p50', df['val_no2'] >= p_no2.get(50, 0)),
            ('all_valid', df['val_no2'].notna()),
        ]
    elif clase == 'high_SO2_pollution':
        thresholds = [
            ('p90', df['val_so2'] >= p_so2.get(90, 0)),
            ('p75', df['val_so2'] >= p_so2.get(75, 0)),
            ('p50', df['val_so2'] >= p_so2.get(50, 0)),
            ('all_valid', df['val_so2'].notna()),
        ]
    elif clase == 'anomalous_ozone':
        thresholds = [
            ('p90', df['val_o3'] >= p_o3.get(90, 0)),
            ('p75', df['val_o3'] >= p_o3.get(75, 0)),
            ('p50', df['val_o3'] >= p_o3.get(50, 0)),
            ('all_valid', df['val_o3'].notna()),
        ]
    elif clase == 'dense_vegetation':
        thresholds = [
            ('p50_clean+lat>3.40',
             (df['val_no2'] < p_no2.get(50, df['val_no2'].median())) &
             (df['val_so2'] < p_so2.get(50, df['val_so2'].median())) &
             (df['lat_centroid'] > 3.40)),
            ('p75_clean+lat>3.35',
             (df['val_no2'] < p_no2.get(75, df['val_no2'].quantile(0.75))) &
             (df['lat_centroid'] > 3.35)),
            ('all_high_lat', df['lat_centroid'] > 3.35),
            ('all_valid', pd.Series(True, index=df.index)),
        ]
    else:  # urban_land
        thresholds = [
            ('p50_clean+lat<=3.50',
             (df['val_no2'] < p_no2.get(50, df['val_no2'].median())) &
             (df['val_so2'] < p_so2.get(50, df['val_so2'].median())) &
             (df['lat_centroid'] <= 3.50)),
            ('all_valid_lat<=3.55',
             df['lat_centroid'] <= 3.55),
            ('all_valid', pd.Series(True, index=df.index)),
        ]

    chosen_subset = None
    chosen_label  = 'all_valid'

    for label, mask in thresholds:
        subset = df[mask & df['val_no2'].notna()].copy()
        n_unicos = subset['idx_zarr'].nunique()
        if n_unicos >= pool_min_unicos:
            chosen_subset = subset
            chosen_label  = label
            break

    if chosen_subset is None:
        # Ultimo recurso: todos los registros validos
        chosen_subset = df[df['val_no2'].notna()].copy()
        if len(chosen_subset) == 0:
            chosen_subset = df.copy()
        chosen_label = 'all_records_fallback'

    n_unicos_pool = chosen_subset['idx_zarr'].nunique()
    replace       = len(chosen_subset) < n_target

    sampled = chosen_subset.sample(
        n=n_target, random_state=42, replace=replace
    ).reset_index(drop=True)

    return sampled, n_unicos_pool, chosen_label


import random
random.seed(42)
np.random.seed(42)

print('Muestreo dirigido con relajacion dinamica (v7):')
all_pairs = []

for clase in CLASES_OBJETIVO:
    class_df, n_unicos, umbral = dynamic_sample_class(
        df_aligned, clase, PARES_POR_CLASE, pctls
    )
    print(f'  {clase}: pool unicos={n_unicos} | umbral="{umbral}" | '
          f'clonado={"SI" if n_unicos < PARES_POR_CLASE else "NO"}')

    for sample_idx, (_, row) in enumerate(class_df.iterrows()):
        fecha_str = str(row['fecha'])[:10] if pd.notna(row['fecha']) else 'unknown'
        pair_id   = f"{clase}_{fecha_str}_{row['idx_zarr']}_{sample_idx:04d}"
        caption   = build_caption(
            clase, pair_id,
            row['val_no2'], row['val_so2'], row['val_o3'],
            row['lat_centroid'], row['lon_centroid'],
            row['fecha'], sample_idx=sample_idx,
        )
        all_pairs.append({
            'tile_id'      : int(row['idx_zarr']),
            'pair_id'      : pair_id,
            'scene_id'     : str(row['escena_id']),
            'date'         : fecha_str,
            'lat_centroid' : float(row['lat_centroid']),
            'lon_centroid' : float(row['lon_centroid']),
            'val_no2'      : float(row['val_no2']) if not np.isnan(row['val_no2']) else None,
            'val_so2'      : float(row['val_so2']) if not np.isnan(row['val_so2']) else None,
            'val_o3'       : float(row['val_o3'])  if not np.isnan(row['val_o3'])  else None,
            'clase'        : clase,
            'caption_en'   : caption,
            'centroides_sinteticos': CENTROIDES_SINTETICOS,
            'umbral_muestreo'      : umbral,
        })

df_final = pd.DataFrame(all_pairs)

print(f'\nTotal pares: {len(df_final):,}')
print(f'  {"OK" if len(df_final) >= 1000 else "FALLA"}: {len(df_final)} pares')
print('\nDistribucion por clase:')
print(df_final['clase'].value_counts().to_string())

n_unicas = df_final['caption_en'].nunique()
print(f'\nCaptions unicas: {n_unicas}/{len(df_final)} ({n_unicas/len(df_final):.1%})')
print(f'  (Objetivo: >80% con ancla temporal hora sintetica)')

# Diagnostico de diversidad de pair_id (cada uno debe ser unico)
n_pair_ids = df_final['pair_id'].nunique()
print(f'pair_ids unicos: {n_pair_ids}/{len(df_final)} ({n_pair_ids/len(df_final):.1%})')


Muestreo dirigido con relajacion dinamica (v7):
  high_NO2_pollution: pool unicos=88 | umbral="p50" | clonado=SI
  high_SO2_pollution: pool unicos=118 | umbral="p50" | clonado=SI
  anomalous_ozone: pool unicos=88 | umbral="p50" | clonado=SI
  dense_vegetation: pool unicos=176 | umbral="all_valid" | clonado=SI
  urban_land: pool unicos=176 | umbral="all_valid_lat<=3.55" | clonado=SI

Total pares: 1,000
  OK: 1000 pares

Distribucion por clase:
clase
high_NO2_pollution    200
high_SO2_pollution    200
anomalous_ozone       200
dense_vegetation      200
urban_land            200

Captions unicas: 1000/1000 (100.0%)
  (Objetivo: >80% con ancla temporal hora sintetica)
pair_ids unicos: 1000/1000 (100.0%)


## 9 · Split Estratificado 70/15/15

In [16]:
SEED = 42

def robust_stratified_split(df, label_col, train_frac=0.70, seed=42):
    from sklearn.model_selection import StratifiedShuffleSplit
    try:
        sss = StratifiedShuffleSplit(n_splits=1, test_size=1-train_frac, random_state=seed)
        i_tr, i_te = next(sss.split(df, df[label_col]))
    except ValueError:
        rng  = np.random.RandomState(seed)
        i_tr = rng.choice(len(df), size=int(len(df)*train_frac), replace=False)
        i_te = np.setdiff1d(np.arange(len(df)), i_tr)
    df_train = df.iloc[i_tr].copy()
    df_temp  = df.iloc[i_te].copy()
    try:
        sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
        iv, it = next(sss2.split(df_temp, df_temp[label_col]))
        df_val  = df_temp.iloc[iv].copy()
        df_test = df_temp.iloc[it].copy()
    except ValueError:
        rng = np.random.RandomState(seed)
        idx = np.arange(len(df_temp)); rng.shuffle(idx)
        half = len(idx)//2
        df_val  = df_temp.iloc[idx[:half]].copy()
        df_test = df_temp.iloc[idx[half:]].copy()
    return df_train, df_val, df_test


df_train, df_val, df_test = robust_stratified_split(df_final, 'clase', seed=SEED)
df_final['split'] = 'train'
df_final.loc[df_val.index,  'split'] = 'val'
df_final.loc[df_test.index, 'split'] = 'test'

print(f'Split 70/15/15 (seed={SEED}):')
print(f'  Train: {len(df_train):,} ({len(df_train)/len(df_final):.1%})')
print(f'  Val  : {len(df_val):,}  ({len(df_val)/len(df_final):.1%})')
print(f'  Test : {len(df_test):,} ({len(df_test)/len(df_final):.1%})')
print('\nClases en train:')
print(df_train['clase'].value_counts().to_string())


Split 70/15/15 (seed=42):
  Train: 699 (69.9%)
  Val  : 150  (15.0%)
  Test : 151 (15.1%)

Clases en train:
clase
high_NO2_pollution    140
urban_land            140
anomalous_ozone       140
dense_vegetation      140
high_SO2_pollution    139


## 10 · Exportar Dataset y MD5

In [17]:
with open(OUTPUT_JSONL, 'w', encoding='utf-8') as f:
    for _, row in df_final.iterrows():
        rec = {k: (None if isinstance(v, float) and np.isnan(v) else v)
               for k, v in row.to_dict().items()}
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

csv_path = OUTPUT_JSONL.with_suffix('.csv')
df_final.to_csv(csv_path, index=False, encoding='utf-8')

def md5_file(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''): h.update(chunk)
    return h.hexdigest()

summary = {
    'fix_version'         : 'v9 - cache KDTree por pollutant + NEG_THRESHOLD + sin parches sinteticos',
    'n_pairs'             : len(df_final),
    'n_clases'            : df_final['clase'].nunique(),
    'clases'              : df_final['clase'].value_counts().to_dict(),
    'n_captions_unicas'   : int(df_final['caption_en'].nunique()),
    'pct_captions_unicas' : float(df_final['caption_en'].nunique() / len(df_final)),
    'centroides_sinteticos': bool(CENTROIDES_SINTETICOS),
    'neg_threshold'       : NEG_THRESHOLD,
    'split'               : {'train': len(df_train), 'val': len(df_val), 'test': len(df_test)},
    'nan_fraction'        : {
        'no2': float(df_final['val_no2'].isna().mean()),
        'so2': float(df_final['val_so2'].isna().mean()),
        'o3' : float(df_final['val_o3'].isna().mean()),
    },
    'pctls_NO2': pctls.get('NO2', {}),
    'pctls_SO2': pctls.get('SO2', {}),
    'pctls_O3' : pctls.get('O3',  {}),
    'md5_jsonl': md5_file(OUTPUT_JSONL),
    'md5_csv'  : md5_file(csv_path),
}

SUMMARY_PATH = Path(r'D:\\analitica\\s5p_alignment_summary_v9.json')
with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('\n-- Resumen Final v9 --')
print(json.dumps(summary, indent=2, ensure_ascii=False))

# Criterios de aceptacion v9
assert len(df_final) >= 1000, f'FALLA: {len(df_final)} < 1000'
for c in CLASES_OBJETIVO:
    n = (df_final['clase'] == c).sum()
    assert n > 0, f'FALLA: clase {c} = 0 muestras'
n_unicas = df_final['caption_en'].nunique()
assert n_unicas / len(df_final) >= 0.80, \
    f'FALLA: captions unicas = {n_unicas/len(df_final):.1%} < 80%'

# Criterio v9 adicional: percentiles de NO2 y SO2 deben diferir
no2_p50 = pctls.get('NO2',{}).get(50, None)
so2_p50 = pctls.get('SO2',{}).get(50, None)
if no2_p50 is not None and so2_p50 is not None and abs(no2_p50 - so2_p50) < 1e-12:
    print('\n[WARN v9] NO2 p50 == SO2 p50. Revisar upload Azure NB01 (blobs identicos en origen).')
else:
    print('\n[OK v9] NO2 y SO2 con distribuciones diferenciadas.')

print('\nCRITERIOS DE ACEPTACION NB02 v9: TODOS CUMPLIDOS')



-- Resumen Final v9 --
{
  "fix_version": "v9 - cache KDTree por pollutant + NEG_THRESHOLD + sin parches sinteticos",
  "n_pairs": 1000,
  "n_clases": 5,
  "clases": {
    "high_NO2_pollution": 200,
    "high_SO2_pollution": 200,
    "anomalous_ozone": 200,
    "dense_vegetation": 200,
    "urban_land": 200
  },
  "n_captions_unicas": 1000,
  "pct_captions_unicas": 1.0,
  "centroides_sinteticos": false,
  "neg_threshold": {
    "NO2": -5e-05,
    "SO2": -0.001,
    "O3": -0.01
  },
  "split": {
    "train": 699,
    "val": 150,
    "test": 151
  },
  "nan_fraction": {
    "no2": 0.0,
    "so2": 0.0,
    "o3": 0.0
  },
  "pctls_NO2": {
    "10": 0.0,
    "25": 0.0,
    "50": 1.4251736507597012e-05,
    "75": 2.3891574918106197e-05,
    "90": 3.140335493481275e-05,
    "99": 4.281552026754477e-05
  },
  "pctls_SO2": {
    "10": -0.0002209381414395162,
    "25": -9.492214014401823e-05,
    "50": 0.0,
    "75": 7.850251234810288e-05,
    "90": 0.0005123385926711191,
    "99": 0.0008293830